In [ ]:
import json
import sys
import re
import pandas as pd
from rich import print as rp
from collections import Counter
from pathlib import Path
from datetime import datetime

from splink import Linker, DuckDBAPI, SettingsCreator, block_on
import splink.comparison_library as cl
from splink.blocking_analysis import count_comparisons_from_blocking_rule


nb_dir = Path.cwd()
project_root = nb_dir.parent.parent
sys.path.insert(0, str(project_root))

In [15]:
people_file = Path(project_root / "data_reload/db_exports/people-2026-08-13.json")

people_df = pd.DataFrame((json.load(open(people_file))))
people_df["unique_id"] = people_df.index

rp(len(people_df))

8936

In [14]:
db_api = DuckDBAPI()

settings = SettingsCreator(
    link_type="dedupe_only",
    blocking_rules_to_generate_predictions=[
        block_on("family_name"),
        block_on("family_name", "substr(given_names, 1, 1)"),
    ],
    comparisons=[
        cl.NameComparison("family_name"),
        cl.NameComparison("given_names"),
    ],
    retain_intermediate_calculation_columns=True
)

# br = block_on("substr(given_names, 1, 1)", "family_name")

# count_comparisons_from_blocking_rule(
#     table_or_tables=people_df,
#     blocking_rule=br,
#     link_type="dedupe_only",
#     db_api=db_api
# )


linker = Linker([people_df], settings, db_api=db_api)

linker.training.estimate_probability_two_random_records_match([block_on("family_name")], recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=1e8)
linker.training.estimate_parameters_using_expectation_maximisation(block_on("family_name"))
linker.training.estimate_parameters_using_expectation_maximisation(block_on("given_names"))

linker.misc.save_model_to_json("splink_model.json", overwrite=True)

# linker.visualisations.parameter_estimate_comparisons_chart()


Probability two random records match is estimated to be  0.000179.
This means that amongst all possible pairwise record comparisons, one in 5,595.74 are expected to match.  With 39,921,580 total possible comparisons, we expect a total of around 7,134.29 matching pairs
----- Estimating u probabilities using random sampling -----


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - family_name (no m values are trained).
    - given_names (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."family_name" = r."family_name"

Parameter estimates will be made for the following comparison(s):
    - given_names

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - family_name

Iteration 1: Largest change in params was -0.832 in the m_probability of given_names, level `Exact match on given_names`
Iteration 2: Largest change in params was 0.093 in the m_probability of given_names, level `All other comparisons`
Iteration 3: Largest change in params was 0.0411 in the m_probability of given_names, level `All other comparisons`
Iteration 4: Largest change in params was -0.0228 in the m_probability of given_names, level `J

{'link_type': 'dedupe_only',
 'probability_two_random_records_match': 0.00017870749890875348,
 'retain_matching_columns': True,
 'retain_intermediate_calculation_columns': True,
 'additional_columns_to_retain': [],
 'sql_dialect': 'duckdb',
 'linker_uid': 'jlrmg3jk',
 'em_convergence': 0.0001,
 'max_iterations': 25,
 'bayes_factor_column_prefix': 'bf_',
 'term_frequency_adjustment_column_prefix': 'tf_',
 'comparison_vector_value_column_prefix': 'gamma_',
 'unique_id_column_name': 'unique_id',
 'source_dataset_column_name': 'source_dataset',
 'blocking_rules_to_generate_predictions': [{'blocking_rule': 'l."family_name" = r."family_name"',
   'sql_dialect': 'duckdb'},
  {'blocking_rule': '(l."family_name" = r."family_name") AND (SUBSTRING(l.given_names, 1, 1) = SUBSTRING(r.given_names, 1, 1))',
   'sql_dialect': 'duckdb'}],
 'comparisons': [{'output_column_name': 'family_name',
   'comparison_levels': [{'sql_condition': '"family_name_l" IS NULL OR "family_name_r" IS NULL',
     'label_fo

In [ ]:
# predictions = linker.inference.predict()

# edges = predictions.as_pandas_dataframe()
# edges = edges.sort_values("match_weight", ascending=False)
# records = edges.head(20).to_dict(orient="records")
# linker.visualisations.waterfall_chart(records)

# edges

Blocking time: 0.01 seconds
Predict time: 0.07 seconds


alt.LayerChart(...)